# 데이터 정제(Data Cleaning)
- 실제 현업 데이터는 빈칸(결측치)이나 오타가 정말 많음. 이 과정을 잘해야 "쓰레기를 넣으면 쓰레기가 나온다(GIGO)"는 상황을 피할 수 있음.

#### 데이터 정제 핵심 개념
1. 결측치(NaN) 처리: 데이터에 값이 비어있는 상태를 NaN이라고 함
- dropna(): 비어있는 줄을 시원하게 삭제
- fillna(): 비어있는 칸을 평균값이나 '0'같은 특정 값으로 채움
2. 중복값 제거(drop_duplicates): 똑같은 데이터가 두 번 들어갔을 때 하나만 남기고 지움
3. 타입 변환(astype): 숫자가 문자열로 저장되어 있으면 계산이 안 됨. 이를 정수(int)나 실수(flaot)로 바꿔줌
4. 문자열 처리(str): 글자 앞뒤 공백을 지우거나(strip), 특정 글자를 바꾸는(replace)등 엑셀의 텍스트 함수 같은 기능을 함
5. 이상값(Outlier) 검증: 말도 안 되게 크거나 작은 데이터(예: 나이가 500살)를 찾아내는 과정

In [2]:
# [Pandas 데이터 정제 - Clean Data 만들기]

import pandas as pd
import numpy as np

# 1.오류가 포함된 데이터 생성
data = {
    "고객명": ["Bob", "Alice", "Bob ", "Charlie", "David"],
    "나이": [25, 30, 25, np.nan, 200], # 결측치와 이상값 포함
    "가입일": ["2026-01-01", "2026-01-02", "2026-01-01", "2026-01-03", "2026-01-04"],
    "구매금액": ["10000", "20000", "10000", "15000", "missing"] #문자열과 오류값
}
df = pd.DataFrame(data)

# 2.중복 데이터 제거 및 문자열 공백 제거
df["고객명"] = df["고객명"].str.strip() # "Bob " -> "Bob"
df = df.drop_duplicates() # 중복된 Bob 데이터 한 줄 삭제

# 3.결측치(NaN) 처리
# 나이의 빈칸을 평균값으로 채우기 (이상값 200 제외 전 평균이라 높을 수 있음)
df["나이"] = df["나이"].fillna(df["나이"].mean() if "나이" in df else 0)

# 4.데이터 타입 변환 및 오류 처리
# 'missing'같은 글자를 0으로 바꾸고 숫자로 변환
df["구매금액"] = df["구매금액"].replace("missing", "0")
df["구매금액"] = df["구매금액"].astype(int)

# 5.데이터 검증 (이상값 처리)
# 나이가 100살 이상인 데이터는 잘못된 것으로 판단하여 필터링
df = df[df["나이"] < 100]

print("--- 정제된 데이터프레임 ---")
print(df)
print("\n--- 데이터 타입 확인 ---")
print(df.dtypes)

--- 정제된 데이터프레임 ---
       고객명    나이         가입일   구매금액
0      Bob  25.0  2026-01-01  10000
1    Alice  30.0  2026-01-02  20000
3  Charlie  85.0  2026-01-03  15000

--- 데이터 타입 확인 ---
고객명      object
나이      float64
가입일      object
구매금액      int64
dtype: object


**[핵심 요약]**
1. 결측치(NaN): 일기장에 '깜빡하고 안 쓴 칸'암. 지우개로 지우거나(dropna), 대출 비슷한 내용을 써넣는(fillna) 것
2. 중복값: 똑같은 스티커가 두 장이 있으면 한장은 떼서 버리자! (drop_duplicates)
3. 타입 변환: 상자 겉면에는 "사과"라고 적혀 있는데 안에는 "배"가 들어있는 상황. 내용물에 맞게 '상자 이름표'를 다시 붙여 주는 것 (astype)
4. 이상값: 우리 반 친구들 몸무게를 재는데 값자기 '공룡 몸무게'가 나온 것. 이건 측정이 잘못된 거니깐 빼고 계산이 맞다!

#### 중급 핵심 문법: 똑똑한 데이터 변환
1. apply(): 나만의 공식 적용하기
- 데이터프레임의 행이나 열에 내가 만든 함수(Function)를 통째로 적용. 복잡한 계산식이나 조건문이 필요할 때 가장 많이 쓰임
2. map(): 일대일 치환하기
- 딕셔너리를 이용해 특정 값을 다른 값으로 1:1 대응시켜 바꿈 (예: '남' -> 1, '여' -> 0)
3. np.where(): 조건부 치환
- 엑셀의 IF 함수와 같음. "조건이 참이면 A, 거짓이면 B를 넣어라"를 한 줄로 끝낼 수 있음

In [3]:
# [Pandas 중급 - 조건부 치환과 함수 적용]

import pandas as pd
import numpy as np

# 1.원본 데이터
data = {
    "고객명": ["Bob", "Alice", "Charlie", "David", "Eve"],
    "구매금액": [150000, 45000, 200000, 10000, 80000],
    "지역코드": [1, 2, 1, 3, 2]
}
df = pd.DataFrame(data)

# 2.apply()를 활용한 등급 분류 (함수 정의)
def classify_grade(amount):
    if amount >= 100000:
        return "VIP"
    elif amount >= 50000:
        return "GOLD"
    else:
        return "SILVER"

df["고객등급"] = df["구매금액"].apply(classify_grade)

# 3.map()을 활용한 지역명 치환 (딕셔너리 활용)
region_map = {1: "서울", 2: "경기", 3: "부산"}
df["지역명"] = df["지역코드"].map(region_map)

# 4.np.where()를 활용한 타겟팅 여부 (엑셀 IF 스타일)
# 구매금액이 5만원 이상이면 '대상', 아니면 '제외'
df["마케팅대상"] = np.where(df["구매금액"] >= 50000, "대상", "제외")

print("--- 데이터 변환 결과 ---")
print(df)

--- 데이터 변환 결과 ---
       고객명    구매금액  지역코드    고객등급 지역명 마케팅대상
0      Bob  150000     1     VIP  서울    대상
1    Alice   45000     2  SILVER  경기    제외
2  Charlie  200000     1     VIP  서울    대상
3    David   10000     3  SILVER  부산    제외
4      Eve   80000     2    GOLD  경기    대상


**[핵심 요약]**
1. apply(): '자동 요리 기계'임. 내가 "재료(데이터)를 넣으면 껍질을 까고 채를 썰어라(함수)"라고 명령어를 입력하면, 기계가 모든 재료를 똑같은 방식으로 손직해 주는 것
2. map(): '비밀번호 해독기'임. "1번은 서울이야. 2번은 경기야"라고 적힌 종이를 주면, 숫자를 보고 바로 글자로 바꿔줌
3. np.where(): 'OX 퀴즈 심판'임. "너 5만원 넘게 썼니?" 라고 물어보고, "네" 라고 하면 합격도장을, "아니오"라고 하면 불합격 도장을 꽝 찍어주는 것

#### 고급 핵심 문법: 데이터의 입체적 분석
1. 피벗 테이블(pivot_table): 수많은 데이터를 내가 원하는 행과 열로 재구성하여 통계를 냄. "날짜별, 지역별 매출 합계" 처럼 보고서 양식을 만들 때 필수
2. 시계열 데이터 처리(to_datetime): 글자로 된 날짜("2026-05-13")를 파이썬이 이해하는 '시간 객체'로 바꿈. 이렇게 하면 "요일별 추출", "월별 합산"이 클릭 한 번으로 가능해짐

In [4]:
# [고급 핵심 문법 - 시계열 분석과 피벗 테이블]

import pandas as pd

# 1.원본 데이터 (날짜가 문자열인 상태)

data = {
    "날짜": ["2026-05-01", "2026-05-01", "2026-05-02", "2026-05-02", "2026-05-03"],
    "채널": ["Google", "Facebook", "Google", "Facebook", "Google"],
    "매출": [150000, 200000, 180000, 150000, 300000],
    "클릭수": [450, 600, 500, 400, 900]
}

df = pd.DataFrame(data)

# 2.시계열 데이터 변환
df["날짜"] = pd.to_datetime(df["날짜"])

# 날짜에서 '요일' 정보만 추출하기 (0=월, 1=화, ...)
df["요일"] = df["날짜"].dt.day_name()

# 3.피벗 테이블 생성
# 행은 '채널', 열은 '요일', 값은 '매출'의 합계(sum)로 요약
pivot_report = df.pivot_table(
    values="매출",
    index="채널",
    columns="요일",
    aggfunc="sum",
    fill_value=0 # 데이터가 없는 칸은 0으로 채우기
)

print("--- 요일별/채널별 매출 피벗 리포트 ---")
print(pivot_report)

# 4.시계열 그룹화: 월별 합계 (예시는 모두 5월이지만 사용법 확인)
monthly_summary = df.resample('ME', on='날짜').sum(numeric_only=True)
print("\n--- 월별 성과 요약 ---")
print(monthly_summary)

--- 요일별/채널별 매출 피벗 리포트 ---
요일        Friday  Saturday  Sunday
채널                                
Facebook  200000    150000       0
Google    150000    180000  300000

--- 월별 성과 요약 ---
                매출   클릭수
날짜                      
2026-05-31  980000  2850


**[핵심 요약]**
1. 시계열 데이터(to_datetime): 그냥 '2026년 5월 1일'이라고 적힌 '글자'를 파이썬에게 주면서 "이건 '달력'이야!"라고 알려주는 것. 그래야 파이썬이 "아, 이날은 금요일이었구나!"라고 똑똑하게 대답할 수 있음
2. 피벗 테이블: 흩어져 있는 '레고 블럭'들을 색깔별(채널), 모양별(요일)로 다시 '예쁘게 쌀아 올리는 것'과 같음. 복잡한 데이터를 한 눈에 들어오는 표로 만드는 마법